# Model 3: Generative LLM (advanced/exploratory)

In this model, full text reviews are fed to a generative LLM which “reads” the reviews and returns aspect-sentiment pairs in a defined JSON format. The OpenAI model `gpt-4.1-mini` was selected because it offers strong performance on structured extraction tasks, reliable instruction following, and significantly lower cost and latency than larger models, making it practical for processing thousands of reviews via the API.

## OpenAI API Key

This notebook requires an OpenAI API key.

When running the notebook, you will be prompted to enter your key.

You can obtain a key from:
https://platform.openai.com/api-keys

Approximate cost and time of each API call is reported in the markdown text before the call.

In [1]:
# Install/upgrade the OpenAI Python client
%pip install -U openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Securely request OpenAI API Key from notebook user
import os
from getpass import getpass
from openai import OpenAI

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

# Select OpenAI model and prompt lag time
MODEL = "gpt-4.1-mini"
SLEEP_SECONDS = 0.2

Enter your OpenAI API key:  ········


## Imports

In [4]:
import json
import time
import random
import warnings
import pandas as pd
from collections import defaultdict

from tqdm import tqdm
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report
)

warnings.filterwarnings("ignore")

## A. Weak Supervision Performance

First, the model is analyzed using the data labeled for weak supervision. This data contains the full review text, as well as, the weakly-assigned ABSA labels. See the data processing (`01_process_data.ipynb`) and feature engineering (`03_feature_engineering.ipynb`) notebooks for more details about this labeling.


### A.1. Load Data

The cleaned, full-review, weakly-labeled dataset is read from the CSV source in chunks to avoid loading the entire dataset into memory.

A smaller subset of reviews is then selected for analysis due to the computational time and API cost associated with prompting a generative LLM.

The subset is intentionally constructed to be balanced across rating levels and to limit the influence of any single business by allowing no more than three reviews per business (`gmap_id`). Additionally, reviews that were selected for manual ABSA labelling are removed to prevent leakage.

In [ ]:
# ---------------------------------------------------------------------------------------
# DATA LOADING SETTINGS
# ---------------------------------------------------------------------------------------

# Define raw data file & manually-labelled ABSA training data
INFILE = "../data/cleaned_for_LLM.csv"
ABSA_FILE = "../data/absa_training_set.csv"

# Select chunk size and sample size, also random state so that the same dataset is generated if the code is rerun
SAMPLE_SIZE = 2500
CHUNK_SIZE = 750000
RANDOM_STATE = 120

# Column labels
LABEL_COLS = [
    "product_quality_positive",
    "product_quality_negative",
    "service_positive",
    "service_negative",
    "wait_time_positive",
    "wait_time_negative",
    "price_value_positive",
    "price_value_negative",
    "cleanliness_positive",
    "cleanliness_negative",
    "atmosphere_positive",
    "atmosphere_negative",
    "general_positive",
    "general_negative",
]

USECOLS = ["review_id", "rating", "text", "gmap_id"] + LABEL_COLS

# Save paths
EVAL_SAMPLE_OUTFILE = "../data/openai_eval_sample.csv"
ZERO_RESULTS_OUTFILE = "../data/openai_zero_shot_results.csv"
FEW_RESULTS_OUTFILE = "../data/openai_few_shot_results.csv"
COMPARISON_OUTFILE = "../data/openai_comparison_metrics.csv"

In [ ]:
# ------------------------------------------------------------------------
# DATA LOADING / SAMPLING FUNCTION
# ------------------------------------------------------------------------

def read_balanced_sample_by_rating_and_gmap(
    infile,
    absa_file,
    usecols,
    sample_size,
    chunk_size,
    random_state,
    allowed_ratings=(1, 2, 3, 4, 5),
    max_per_gmap_per_rating=3
):
    """
    Read a large CSV in chunks, exclude review_ids from the manual ABSA set,
    and build a subset that is:
      1) balanced across ratings
      2) spread across gmap_id values within each rating bucket
    """
    random.seed(random_state)

    allowed_ratings = list(allowed_ratings)
    n_ratings = len(allowed_ratings)

    target_per_rating = sample_size // n_ratings
    remainder = sample_size % n_ratings

    target_counts = {r: target_per_rating for r in allowed_ratings}
    for r in allowed_ratings[:remainder]:
        target_counts[r] += 1

    print("Target counts by rating:")
    for r in allowed_ratings:
        print(f"  Rating {r}: {target_counts[r]:,}")

    # Load manual ABSA review_ids to exclude
    absa_ids = set(
        pd.read_csv(absa_file, usecols=["review_id"])["review_id"].astype(str)
    )
    print(f"\nLoaded {len(absa_ids):,} ABSA review_ids to exclude")

    pools = {r: [] for r in allowed_ratings}
    gmap_counts = {r: defaultdict(int) for r in allowed_ratings}
    valid_seen = {r: 0 for r in allowed_ratings}

    reader = pd.read_csv(
        infile,
        usecols=usecols,
        chunksize=chunk_size
    )

    for i, chunk in enumerate(reader, start=1):
        chunk["review_id"] = chunk["review_id"].astype(str)
        chunk["gmap_id"] = chunk["gmap_id"].astype(str)

        # Filter out manually labeled reviews
        chunk = chunk[~chunk["review_id"].isin(absa_ids)]

        # Keep only selected ratings
        chunk = chunk[chunk["rating"].isin(allowed_ratings)]

        if len(chunk) == 0:
            print(f"Chunk {i}: 0 valid rows after filtering")
            continue

        # Shuffle within chunk
        chunk = chunk.sample(frac=1, random_state=random_state + i).reset_index(drop=True)

        for row in chunk.to_dict(orient="records"):
            r = row["rating"]
            g = row["gmap_id"]

            valid_seen[r] += 1

            if gmap_counts[r][g] < max_per_gmap_per_rating:
                pools[r].append(row)
                gmap_counts[r][g] += 1

        msg = [f"Chunk {i}"]
        for r in allowed_ratings:
            msg.append(
                f"r{r}: seen={valid_seen[r]:,}, kept={len(pools[r]):,}, gmaps={len(gmap_counts[r]):,}"
            )
        print(" | ".join(msg))

    final_parts = []

    print("\nFinal bucket sizes before downsampling:")
    for r in allowed_ratings:
        print(f"  Rating {r}: {len(pools[r]):,}")

    for r in allowed_ratings:
        bucket = pools[r]
        target = target_counts[r]

        if len(bucket) == 0:
            print(f"WARNING: Rating {r} has 0 rows after filtering")
            continue

        if len(bucket) >= target:
            bucket_df = pd.DataFrame(bucket).sample(n=target, random_state=random_state)
        else:
            print(
                f"WARNING: Rating {r} only has {len(bucket):,} rows available "
                f"(target was {target:,})"
            )
            bucket_df = pd.DataFrame(bucket)

        final_parts.append(bucket_df)

    df_sample = pd.concat(final_parts, ignore_index=True)
    df_sample = df_sample.sample(frac=1, random_state=random_state).reset_index(drop=True)

    print("\nDone.")
    print(f"Final sample size: {len(df_sample):,}\n")

    print("Final counts by rating:")
    print(df_sample["rating"].value_counts().sort_index())

    print("\nUnique gmap_id counts by rating:")
    print(df_sample.groupby("rating")["gmap_id"].nunique().sort_index())

    return df_sample

In [ ]:
# ------------------------------------------------------------
# Build Subset & Save to .CSV
# ------------------------------------------------------------

df_sample = read_balanced_sample_by_rating_and_gmap(
    infile=INFILE,
    absa_file=ABSA_FILE,
    usecols=USECOLS,
    sample_size=SAMPLE_SIZE,
    chunk_size=CHUNK_SIZE,
    random_state=RANDOM_STATE,
    allowed_ratings=(1, 2, 3, 4, 5),
    max_per_gmap_per_rating=3
)

df_sample.to_csv(EVAL_SAMPLE_OUTFILE, index=False)
print(f"\nSaved evaluation sample to: {EVAL_SAMPLE_OUTFILE}")

display(df_sample.head(3))

### A.2. Generative LLM Prompt Creation

In [ ]:
# -----------------------------------------------------------
# Define Output JSON
# ------------------------------------------------------------

ASPECTS = [
    "Product Quality",
    "Service",
    "Wait Time",
    "Price/Value",
    "Cleanliness",
    "Atmosphere",
    "General"
]

SENTIMENTS = ["positive", "negative", "neutral", "mixed"]

ABSA_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "absa_review_output",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "review_id": {"type": "string"},
                "pairs": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "aspect": {"type": "string", "enum": ASPECTS},
                            "sentiment": {"type": "string", "enum": SENTIMENTS},
                            "evidence": {"type": "string"}
                        },
                        "required": ["aspect", "sentiment", "evidence"],
                        "additionalProperties": False
                    }
                }
            },
            "required": ["review_id", "pairs"],
            "additionalProperties": False
        }
    }
}

In [ ]:
# ----------------------------------------------------------------------------
# Define Prompt
# ----------------------------------------------------------------------------

DEVELOPER_PROMPT = """
You are an expert annotator for aspect-based sentiment analysis (ABSA).

Task:
Read one customer review and extract all aspect-sentiment pairs present.

Allowed aspects:
- Product Quality
- Service
- Wait Time
- Price/Value
- Cleanliness
- Atmosphere
- General

Allowed sentiment labels:
- positive
- negative
- neutral
- mixed

Rules:
1. Return only the requested JSON schema.
2. Include one entry per distinct aspect mentioned in the review.
3. Do not invent aspects not supported by the review.
4. Use General if there is an overall sentiment that does not clearly fit another aspect.
5. Keep evidence short and directly grounded in the review text.
6. If an aspect contains both positive and negative sentiment, use mixed.
7. If there is no clear aspect sentiment, return an empty list for pairs.
"""

In [ ]:
# --------------------------------------------------------------------------
# FEW-SHOT EXAMPLES
# --------------------------------------------------------------------------

FEW_SHOT_EXAMPLES = [
    {
        "role": "user",
        "content": "Review ID: ex1\nReview: The tacos were amazing and cheap, but the dining room was dirty."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex1",
            "pairs": [
                {"aspect": "Product Quality", "sentiment": "positive", "evidence": "tacos were amazing"},
                {"aspect": "Price/Value", "sentiment": "positive", "evidence": "cheap"},
                {"aspect": "Cleanliness", "sentiment": "negative", "evidence": "dining room was dirty"}
            ]
        })
    },
    {
        "role": "user",
        "content": "Review ID: ex2\nReview: Our server was kind, but we waited 40 minutes for the food."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex2",
            "pairs": [
                {"aspect": "Service", "sentiment": "positive", "evidence": "server was kind"},
                {"aspect": "Wait Time", "sentiment": "negative", "evidence": "waited 40 minutes"}
            ]
        })
    },
    {
        "role": "user",
        "content": "Review ID: ex3\nReview: Great place overall. I'll definitely come back."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex3",
            "pairs": [
                {"aspect": "General", "sentiment": "positive", "evidence": "Great place overall"}
            ]
        })
    },
    {
        "role": "user",
        "content": "Review ID: ex4\nReview: The restaurant looked nice but the tables were sticky."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex4",
            "pairs": [
                {"aspect": "Atmosphere", "sentiment": "positive", "evidence": "restaurant looked nice"},
                {"aspect": "Cleanliness", "sentiment": "negative", "evidence": "tables were sticky"}
            ]
        })
    },
    {
        "role": "user",
        "content": "Review ID: ex5\nReview: The burger was okay and the price was fair."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex5",
            "pairs": [
                {"aspect": "Product Quality", "sentiment": "neutral", "evidence": "burger was okay"},
                {"aspect": "Price/Value", "sentiment": "neutral", "evidence": "price was fair"}
            ]
        })
    },
    {
        "role": "user",
        "content": "Review ID: ex6\nReview: The staff were friendly but also forgot our drinks several times."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex6",
            "pairs": [
                {"aspect": "Service", "sentiment": "mixed", "evidence": "staff were friendly but forgot our drinks"}
            ]
        })
    },
    {
        "role": "user",
        "content": "Review ID: ex7\nReview: The line moved quickly and the food came out fast."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex7",
            "pairs": [
                {"aspect": "Wait Time", "sentiment": "positive", "evidence": "line moved quickly"}
            ]
        })
    }
]

### A.3. Build Prompts

In [ ]:
# Function to build prompt
def build_messages(review_id, review_text, mode="zero_shot"):
    messages = [{"role": "developer", "content": DEVELOPER_PROMPT}]

    if mode == "few_shot":
        messages.extend(FEW_SHOT_EXAMPLES)

    messages.append({
        "role": "user",
        "content": f"Review ID: {review_id}\nReview: {review_text}"
    })

    return messages

In [ ]:
# Function to call API
def call_absa_model(review_id, review_text, mode="zero_shot", max_retries=3):
    """
    Send one review to OpenAI and return parsed structured output.
    Also returns token usage for later cost analysis.
    """
    messages = build_messages(review_id, review_text, mode=mode)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                response_format=ABSA_SCHEMA,
                temperature=0
            )

            content = response.choices[0].message.content
            parsed = json.loads(content)
            usage = response.usage

            if "review_id" not in parsed:
                parsed["review_id"] = str(review_id)
            if "pairs" not in parsed or parsed["pairs"] is None:
                parsed["pairs"] = []

            return {
                "review_id": str(review_id),
                "success": True,
                "parsed": parsed,
                "error": None,
                "prompt_tokens": getattr(usage, "prompt_tokens", None),
                "completion_tokens": getattr(usage, "completion_tokens", None),
                "total_tokens": getattr(usage, "total_tokens", None),
            }

        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                return {
                    "review_id": str(review_id),
                    "success": False,
                    "parsed": {"review_id": str(review_id), "pairs": []},
                    "error": str(e),
                    "prompt_tokens": None,
                    "completion_tokens": None,
                    "total_tokens": None,
                }

In [ ]:
# Map API output to 14 ABSA labels

ASPECT_TO_PREFIX = {
    "Product Quality": "product_quality",
    "Service": "service",
    "Wait Time": "wait_time",
    "Price/Value": "price_value",
    "Cleanliness": "cleanliness",
    "Atmosphere": "atmosphere",
    "General": "general"
}

def pairs_to_binary_label_dict(pairs):
    """
    Convert ABSA pairs into the same 14 binary columns as the weak ABSA labels.
    """
    out = {col: 0 for col in LABEL_COLS}

    for pair in pairs:
        aspect = pair.get("aspect")
        sentiment = pair.get("sentiment")

        if aspect not in ASPECT_TO_PREFIX:
            continue

        prefix = ASPECT_TO_PREFIX[aspect]

        if sentiment == "positive":
            out[f"{prefix}_positive"] = 1
        elif sentiment == "negative":
            out[f"{prefix}_negative"] = 1
        elif sentiment == "mixed":
            out[f"{prefix}_positive"] = 1
            out[f"{prefix}_negative"] = 1
        elif sentiment == "neutral":
            pass

    return out

### A.4. Prompt Model via API

In [ ]:
# ------------------------------------------------------------
# FUNCTION FOR MODEL PROMPTING VIA API
# ------------------------------------------------------------

def run_openai_absa(df_input, mode="zero_shot", sleep_seconds=0.2, save_every=100, outfile=None):
    """
    Run OpenAI ABSA extraction over a dataframe and return a dataframe
    with predicted binary label columns added.

    If outfile is provided, partial checkpoints are saved every `save_every` rows.
    """
    results = []

    for idx, (_, row) in enumerate(tqdm(df_input.iterrows(), total=len(df_input)), start=1):
        review_id = str(row["review_id"])
        review_text = str(row["text"])

        result = call_absa_model(
            review_id=review_id,
            review_text=review_text,
            mode=mode,
            max_retries=3
        )

        pred_dict = pairs_to_binary_label_dict(result["parsed"].get("pairs", []))

        out_row = {
            "review_id": review_id,
            "rating": row["rating"],
            "text": row["text"],
            "gmap_id": row["gmap_id"],
            "mode": mode,
            "success": result["success"],
            "error": result["error"],
            "raw_pairs": json.dumps(result["parsed"].get("pairs", [])),
            "prompt_tokens": result["prompt_tokens"],
            "completion_tokens": result["completion_tokens"],
            "total_tokens": result["total_tokens"],
        }

        for col in LABEL_COLS:
            out_row[f"true__{col}"] = int(row[col])

        for col in LABEL_COLS:
            out_row[f"pred__{col}"] = int(pred_dict[col])

        results.append(out_row)

        if outfile is not None and idx % save_every == 0:
            pd.DataFrame(results).to_csv(outfile, index=False)
            print(f"\nCheckpoint saved: {outfile} ({idx:,} rows processed)")

        if sleep_seconds > 0:
            time.sleep(sleep_seconds)

    results_df = pd.DataFrame(results)

    if outfile is not None:
        results_df.to_csv(outfile, index=False)
        print(f"\nFinal saved: {outfile}")

    return results_df

#### A.5. Run Zero-Shot

Using the default parameters provided, this API call took approximately 60 minutes and resulted in XXX input tokens and YYY output tokens.

As of March 2026, the cost of this using `gpt-4.1-mini` is ~$ZZ.

In [ ]:
results_zero = run_openai_absa(
    df_input=df_sample,
    mode="zero_shot",
    sleep_seconds=SLEEP_SECONDS,
    save_every=100,
    outfile=ZERO_RESULTS_OUTFILE
)

#### A.6. Run Few-Shot

Using the default parameters provided, this API call took approximately 60 minutes and resulted in XXX input tokens and YYY output tokens.

As of March 2026, the cost of this using `gpt-4.1-mini` is ~$ZZ.

In [ ]:
results_few = run_openai_absa(
    df_input=df_sample,
    mode="few_shot",
    sleep_seconds=SLEEP_SECONDS,
    save_every=100,
    outfile=FEW_RESULTS_OUTFILE
)

### A.7. Weak Supervision Evaluation

In [ ]:
# -----------------------------------------------------------
# Function for multi-label evaluation of model performance
# ------------------------------------------------------------

def evaluate_multilabel_results(results_df, label_cols):
    """
    Compute:
    - weighted precision
    - weighted recall
    - weighted F1
    - macro F1
    - subset accuracy
    - classification report
    """
    true_cols = [f"true__{c}" for c in label_cols]
    pred_cols = [f"pred__{c}" for c in label_cols]

    y_true = results_df[true_cols].values
    y_pred = results_df[pred_cols].values

    metrics = {
        "precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "classification_report": classification_report(
            y_true,
            y_pred,
            target_names=label_cols,
            zero_division=0
        )
    }

    return metrics


def print_multilabel_results(model_name, metrics):
    """
    Print results in a format that matches your other model output.
    """
    print(f"{model_name}")
    print(f"Test Accuracy:     {metrics['accuracy']:.4f}\n")
    print(f"F1 Score (macro):    {metrics['macro_f1']:.4f}")
    print(f"F1 Score (weighted): {metrics['weighted_f1']:.4f}\n")
    print("Classification Report:")
    print(metrics["classification_report"])

In [ ]:
zero_metrics = evaluate_multilabel_results(results_zero, LABEL_COLS)
few_metrics = evaluate_multilabel_results(results_few, LABEL_COLS)

print_multilabel_results("ZERO-SHOT METRICS", zero_metrics)
print()
print_multilabel_results("FEW-SHOT METRICS", few_metrics)

### A.8. Comparison of Zero-Shot vs. Few-Shot

In [ ]:
comparison = pd.DataFrame([
    metrics_to_row("zero_shot", zero_metrics),
    metrics_to_row("few_shot", few_metrics),
])

display(comparison)

comparison.to_csv(COMPARISON_OUTFILE, index=False)
print(f"\nSaved comparison metrics to: {COMPARISON_OUTFILE}")

## A.9. Token Usage Summary

In [ ]:
# -----------------------------------------------------------
# Function to calculate token usage
# ------------------------------------------------------------

def summarize_usage(results_df, mode_name):
    prompt_tokens = pd.to_numeric(results_df["prompt_tokens"], errors="coerce").fillna(0).sum()
    completion_tokens = pd.to_numeric(results_df["completion_tokens"], errors="coerce").fillna(0).sum()
    total_tokens = pd.to_numeric(results_df["total_tokens"], errors="coerce").fillna(0).sum()

    print(f"\n{mode_name} token usage")
    print(f"  prompt_tokens:     {int(prompt_tokens):,}")
    print(f"  completion_tokens: {int(completion_tokens):,}")
    print(f"  total_tokens:      {int(total_tokens):,}")

summarize_usage(results_zero, "ZERO-SHOT")
summarize_usage(results_few, "FEW-SHOT")

## B. Manual ABSA Evaluation

### B.1. Load and Prepare Manually Labelled ABSA Data

The manually labelled ABSA data is stored in two files: one containing sentence-level review text and metadata, and one containing long-format aspect-sentiment labels. Because labelling was performed at the sentence level, labels are first converted to binary aspect-sentiment columns and then aggregated to the review level so that repeated mentions of the same label within a review are counted only once.

In [ ]:
# ---------------------------------------------------------------------------------------
# MANUAL ABSA DATA FILES
# ---------------------------------------------------------------------------------------

MANUAL_TEXT_FILE = "../data/absa_training_set.csv"
MANUAL_LABELS_FILE = "../data/absa_labels_long.csv"

# Save paths
MANUAL_EVAL_OUTFILE = "../data/manual_absa_eval_reviews.csv"
MANUAL_ZERO_RESULTS_OUTFILE = "../data/openai_manual_zero_shot_results.csv"
MANUAL_FEW_RESULTS_OUTFILE = "../data/openai_manual_few_shot_results.csv"
MANUAL_COMPARISON_OUTFILE = "../data/openai_manual_comparison_metrics.csv"

In [ ]:
# ---------------------------------------------------------------------------------------
# HELPER MAPS FOR MANUAL LABEL CONVERSION
# ---------------------------------------------------------------------------------------

ASPECT_TO_PREFIX = {
    "Product Quality": "product_quality",
    "Service": "service",
    "Wait Time": "wait_time",
    "Price/Value": "price_value",
    "Cleanliness": "cleanliness",
    "Atmosphere": "atmosphere",
    "General": "general"
}

# Normalization for slightly different style in manual file
ASPECT_NORMALIZATION = {
    "food_quality": "Product Quality",
    "service": "Service",
    "wait_time": "Wait Time",
    "price_value": "Price/Value",
    "cleanliness": "Cleanliness",
    "atmosphere": "Atmosphere",
    "general": "General"
}

SENTIMENT_NORMALIZATION = {
    "positive": "positive",
    "negative": "negative"
}

In [ ]:
# ---------------------------------------------------------------------------------------
# FUNCTION TO BUILD REVIEW-LEVEL MANUAL ABSA EVALUATION SET
# ---------------------------------------------------------------------------------------

def build_manual_absa_review_eval_set(text_file, labels_file, label_cols):
    """
    Build a review-level manual ABSA evaluation set.

    Inputs:
      text_file:
        absa_training_set.csv with columns:
        review_id, gmap_id, rating, sentence_id, sentence_text

      labels_file:
        absa_labels_long.csv with columns:
        review_id, sentence_id, aspect, sentiment

    Output:
      one row per review_id with:
      - full review text reconstructed from ordered sentences
      - review metadata
      - 14 binary gold label columns matching LABEL_COLS

    Notes:
      - repeated labels within a review count once
      - positive/negative labels are aggregated with max()
      - if a review has both positive and negative for the same aspect,
        both binary columns are set to 1
    """
# -------------------------------------------------------------------
    # LOAD DATA
    # -------------------------------------------------------------------
    df_text = pd.read_csv(text_file)
    df_labels = pd.read_csv(labels_file)

    # Standardize ids as strings
    for col in ["review_id", "sentence_id"]:
        if col in df_text.columns:
            df_text[col] = df_text[col].astype(str)
        if col in df_labels.columns:
            df_labels[col] = df_labels[col].astype(str)

    if "gmap_id" in df_text.columns:
        df_text["gmap_id"] = df_text["gmap_id"].astype(str)

    # -------------------------------------------------------------------
    # CLEAN / NORMALIZE MANUAL LABELS
    # -------------------------------------------------------------------
    df_labels["aspect"] = (
        df_labels["aspect"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(ASPECT_NORMALIZATION)
    )


    df_labels["sentiment"] = (
        df_labels["sentiment"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(SENTIMENT_NORMALIZATION)
    )

    # Keep only valid rows
    df_labels = df_labels[
        df_labels["aspect"].notna() &
        df_labels["sentiment"].notna()
    ].copy()

    # -------------------------------------------------------------------
    # CONVERT LONG LABELS TO BINARY REVIEW-LEVEL UPDATES
    # -------------------------------------------------------------------

    label_updates = []

    for _, row in df_labels.iterrows():
        review_id = row["review_id"]
        aspect = row["aspect"]
        sentiment = row["sentiment"]

        if aspect not in ASPECT_TO_PREFIX:
            continue

        prefix = ASPECT_TO_PREFIX[aspect]

        if sentiment == "positive":
            label_updates.append((review_id, f"{prefix}_positive", 1))
        elif sentiment == "negative":
            label_updates.append((review_id, f"{prefix}_negative", 1))

    # -------------------------------------------------------------------
    # BUILD REVIEW-LEVEL GOLD LABEL FRAME
    # -------------------------------------------------------------------

    review_keys = df_text[["review_id"]].drop_duplicates().copy()

    for col in label_cols:
        review_keys[col] = 0

    if len(label_updates) > 0:
        df_updates = pd.DataFrame(
            label_updates,
            columns=["review_id", "label_col", "value"]
        ).drop_duplicates()

        df_updates_wide = (
            df_updates
            .pivot_table(
                index="review_id",
                columns="label_col",
                values="value",
                aggfunc="max",
                fill_value=0
            )
            .reset_index()
        )

        df_gold = review_keys.merge(df_updates_wide, on="review_id", how="left", suffixes=("", "__new"))



        for col in label_cols:
            new_col = f"{col}__new"
            if new_col in df_gold.columns:
                df_gold[col] = df_gold[[col, new_col]].max(axis=1)
                df_gold.drop(columns=[new_col], inplace=True)
    else:
        df_gold = review_keys.copy()

    # -------------------------------------------------------------------
    # RECONSTRUCT FULL REVIEW TEXT FROM SENTENCES
    # -------------------------------------------------------------------


    df_text = df_text.copy()

    # Try to sort by sentence_id in numeric order when possible
    df_text["_sentence_order"] = pd.to_numeric(df_text["sentence_id"], errors="coerce")
    df_text = df_text.sort_values(["review_id", "_sentence_order", "sentence_id"])

    df_reviews = (
        df_text.groupby("review_id", as_index=False)
        .agg({
            "gmap_id": "first",
            "rating": "first",
            "sentence_text": lambda x: " ".join(
                s.strip() for s in x.dropna().astype(str) if s.strip()
            )
        })
        .rename(columns={"sentence_text": "text"})
    )

    # -------------------------------------------------------------------
    # MERGE REVIEW TEXT + REVIEW-LEVEL GOLD LABELS
    # -------------------------------------------------------------------
    df_manual_eval = df_reviews.merge(df_gold, on="review_id", how="left")

    for col in label_cols:
        df_manual_eval[col] = df_manual_eval[col].fillna(0).astype(int)

    # -------------------------------------------------------------------
    # SUMMARY
    # -------------------------------------------------------------------
    print("Manual review-level ABSA evaluation set built.")
    print(f"Rows (reviews): {len(df_manual_eval):,}")
    print(f"Unique gmap_id: {df_manual_eval['gmap_id'].nunique():,}")

    print("\nPositive counts per binary label:")
    print(df_manual_eval[label_cols].sum().sort_values(ascending=False))

    return df_manual_eval


In [ ]:
# ------------------------------------------------------------
# BUILD MANUAL REVIEW-LEVEL EVALUATION SET
# ------------------------------------------------------------

df_manual_eval = build_manual_absa_review_eval_set(
    text_file=MANUAL_TEXT_FILE,
    labels_file=MANUAL_LABELS_FILE,
    label_cols=LABEL_COLS
)

df_manual_eval.to_csv(MANUAL_EVAL_OUTFILE, index=False)
print(f"\nSaved manual review-level evaluation set to: {MANUAL_EVAL_OUTFILE}")

display(df_manual_eval.head(3))

### B.2. Run Zero-Shot on the Manual ABSA Review Set

Zero-shot prompting is first applied to the manually labelled review set. Each reconstructed full review is sent to the model, and the returned aspect-sentiment pairs are mapped into the same 14 binary label columns used for evaluation.


In [ ]:
results_manual_zero = run_openai_absa(
    df_input=df_manual_eval,
    mode="zero_shot",
    sleep_seconds=SLEEP_SECONDS,
    save_every=100,
    outfile=MANUAL_ZERO_RESULTS_OUTFILE
)

### B.3. Run Few-Shot on the Manual ABSA Review Set

Few-shot prompting is then applied to the same manually labelled review set using the example review-response pairs defined earlier.

In [ ]:
results_manual_few = run_openai_absa(
    df_input=df_manual_eval,
    mode="few_shot",
    sleep_seconds=SLEEP_SECONDS,
    save_every=100,
    outfile=MANUAL_FEW_RESULTS_OUTFILE
)

### B.4. Evaluate Manual ABSA Results

The zero-shot and few-shot outputs are evaluated against the manually labelled review-level gold labels using subset accuracy, macro F1, weighted F1, and a full classification report.

In [ ]:
manual_zero_metrics = evaluate_multilabel_results(results_manual_zero, LABEL_COLS)
manual_few_metrics = evaluate_multilabel_results(results_manual_few, LABEL_COLS)

print_multilabel_results("MANUAL ABSA ZERO-SHOT METRICS", manual_zero_metrics)
print()
print_multilabel_results("MANUAL ABSA FEW-SHOT METRICS", manual_few_metrics)

### B.5. Compare Zero-Shot vs. Few-Shot on Manual ABSA Set

In [ ]:
# ------------------------------------------------------------
# FUNCTION TO CONVERT METRICS DICT TO A CLEAN COMPARISON ROW
# ------------------------------------------------------------

def metrics_to_row(mode_name, metrics):
    return {
        "mode": mode_name,
        "accuracy": metrics["accuracy"],
        "precision_weighted": metrics["precision"],
        "recall_weighted": metrics["recall"],
        "f1_macro": metrics["macro_f1"],
        "f1_weighted": metrics["weighted_f1"],
    }
manual_comparison = pd.DataFrame([
    metrics_to_row("zero_shot", manual_zero_metrics),
    metrics_to_row("few_shot", manual_few_metrics),
])

display(manual_comparison)

manual_comparison.to_csv(MANUAL_COMPARISON_OUTFILE, index=False)
print(f"\nSaved manual comparison metrics to: {MANUAL_COMPARISON_OUTFILE}")
B.6. Token Usage Summary for Manual ABSA Set
summarize_usage(results_manual_zero, "MANUAL ZERO-SHOT")
summarize_usage(results_manual_few, "MANUAL FEW-SHOT")

In [ ]:
# ------------------------------------------------------------
# REVIEW-LEVEL ERROR ANALYSIS
# Pull best / worst reviews based on row-level multilabel F1
# ------------------------------------------------------------

def add_row_level_scores(results_df, label_cols):
    """
    Add row-level precision, recall, F1, and exact-match flag.

    For each review:
    - precision = TP / (TP + FP)
    - recall    = TP / (TP + FN)
    - f1        = harmonic mean of precision and recall
    - exact_match = 1 if all labels match exactly, else 0
    """
    df = results_df.copy()

    true_cols = [f"true__{c}" for c in label_cols]
    pred_cols = [f"pred__{c}" for c in label_cols]

    row_precisions = []
    row_recalls = []
    row_f1s = []
    row_exact = []
    row_tp = []
    row_fp = []
    row_fn = []

    for _, row in df.iterrows():
        y_true = row[true_cols].astype(int).values
        y_pred = row[pred_cols].astype(int).values

        tp = ((y_true == 1) & (y_pred == 1)).sum()
        fp = ((y_true == 0) & (y_pred == 1)).sum()
        fn = ((y_true == 1) & (y_pred == 0)).sum()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        exact_match = int((y_true == y_pred).all())

        row_tp.append(tp)
        row_fp.append(fp)
        row_fn.append(fn)
        row_precisions.append(precision)
        row_recalls.append(recall)
        row_f1s.append(f1)
        row_exact.append(exact_match)

    df["row_tp"] = row_tp
    df["row_fp"] = row_fp
    df["row_fn"] = row_fn
    df["row_precision"] = row_precisions
    df["row_recall"] = row_recalls
    df["row_f1"] = row_f1s
    df["exact_match"] = row_exact

    return df


def get_true_pred_label_lists(row, label_cols):
    """
    Return lists of active true and predicted labels for one row.
    """
    true_labels = [col for col in label_cols if int(row[f"true__{col}"]) == 1]
    pred_labels = [col for col in label_cols if int(row[f"pred__{col}"]) == 1]
    return true_labels, pred_labels


def build_error_analysis_table(scored_df, label_cols, n=5, best=True):
    """
    Create a compact table of the best or worst rows.

    Ranking:
    - Best: highest row_f1, then highest exact_match, then lowest total mistakes
    - Worst: lowest row_f1, then lowest exact_match, then highest total mistakes
    """
    df = scored_df.copy()

    df["total_errors"] = df["row_fp"] + df["row_fn"]

    if best:
        df_ranked = df.sort_values(
            by=["row_f1", "exact_match", "total_errors"],
            ascending=[False, False, True]
        ).head(n)
    else:
        df_ranked = df.sort_values(
            by=["row_f1", "exact_match", "total_errors"],
            ascending=[True, True, False]
        ).head(n)

    rows = []
    for _, row in df_ranked.iterrows():
        true_labels, pred_labels = get_true_pred_label_lists(row, label_cols)

        rows.append({
            "review_id": row["review_id"],
            "rating": row.get("rating", None),
            "row_f1": round(row["row_f1"], 4),
            "row_precision": round(row["row_precision"], 4),
            "row_recall": round(row["row_recall"], 4),
            "exact_match": row["exact_match"],
            "tp": row["row_tp"],
            "fp": row["row_fp"],
            "fn": row["row_fn"],
            "text": row.get("text", ""),
            "true_labels": ", ".join(true_labels),
            "pred_labels": ", ".join(pred_labels),
            "raw_pairs": row.get("raw_pairs", "")
        })

    return pd.DataFrame(rows)